# Visualization (Topomaps time slider + confronto soggetti)

Unisce:
- `visualize_time_topomap_slider.ipynb`
- (aggiungi qui sotto anche il blocco "4 topomap" che hai in chat)

Data: 2026-03-05


# EEG feature topomap slider (time windows)

Carica un file `subject_XX.pt` **time-resolved** e mostra una topomap (vista dall'alto) con slider sui 5 step temporali.

Assume che nel `.pt` ci siano:
- `X`: (epochs, windows, channels, features)
- `y`: (epochs,)
- `feature_cols`: lista feature
- `ch_names`: lista canali (stesso ordine di X)

Se non hai `ch_names` nel `.pt`, aggiungilo quando crei i tensori.


In [1]:
from pathlib import Path
import sys

project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")
sys.path.insert(0, str(project_root))
print("project_root:", project_root)


project_root: /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech


In [2]:
import numpy as np
import torch
import mne
import matplotlib.pyplot as plt

from ipywidgets import Dropdown, IntSlider, HBox, VBox, Output
from IPython.display import display, clear_output


## Load subject tensor


In [3]:
SUBJECT_ID = 0

pt_path = project_root / "data/processed/subject_tensors/subject_tensors_time" / f"subject_{SUBJECT_ID:02d}.pt"
obj = torch.load(pt_path, map_location="cpu")

X = obj["X"].numpy()
y = obj["y"].numpy()
feature_cols = obj.get("feature_cols", None)
ch_names = obj.get("ch_names", None)

print("loaded:", pt_path.name)
print("X:", X.shape)
print("num classes:", len(np.unique(y)))
print("has feature_cols:", feature_cols is not None)
print("has ch_names:", ch_names is not None)

assert feature_cols is not None, "Manca feature_cols nel .pt"
assert ch_names is not None, "Manca ch_names nel .pt (aggiungilo in fase di creazione tensori)"


loaded: subject_00.pt
X: (550, 5, 59, 40)
num classes: 110
has feature_cols: True
has ch_names: True


## Build MNE Info from ch_names + montage


In [4]:
eloc_path = project_root / "src/io/ebneuro.locs"
montage = mne.channels.read_custom_montage(str(eloc_path))

info = mne.create_info(ch_names=list(ch_names), sfreq=obj.get("fs", 256), ch_types="eeg")

pos = montage.get_positions()
ch_pos = {k: v for k, v in pos["ch_pos"].items() if k in set(ch_names)}
montage_keep = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")
info.set_montage(montage_keep)

print("info channels:", len(info.ch_names))


info channels: 59


## Slider topomap (no flicker)


In [5]:
n_epochs, n_windows, n_ch, n_feat = X.shape
classes = sorted(np.unique(y).tolist())

feat_dd = Dropdown(options=feature_cols, value=feature_cols[0], description="feature")
class_dd = Dropdown(options=classes, value=classes[0], description="class")
win_sl  = IntSlider(value=0, min=0, max=n_windows-1, step=1, description="window")
out = Output()

def draw(window, feature, cls):
    with out:
        clear_output(wait=True)

        fidx = feature_cols.index(feature)
        idx = np.where(y == cls)[0]
        if len(idx) == 0:
            print("no trials for class", cls)
            return

        vals = X[idx, window, :, fidx].mean(axis=0)

        fig, ax = plt.subplots(figsize=(9, 9), dpi=150)
        im, _ = mne.viz.plot_topomap(
            vals,
            info,
            axes=ax,
            show=False,
            sensors=True,
            contours=0,
            outlines="head",
            image_interp="cubic",
        )
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(feature)

        ax.set_title(f"Subject {SUBJECT_ID:02d} | class={cls} | {feature} | window={window}", fontsize=14)
        plt.show()

def _on_change(change=None):
    draw(win_sl.value, feat_dd.value, class_dd.value)

win_sl.observe(_on_change, names="value")
feat_dd.observe(_on_change, names="value")
class_dd.observe(_on_change, names="value")

ui = HBox([win_sl, feat_dd, class_dd])
display(VBox([ui, out]))

draw(win_sl.value, feat_dd.value, class_dd.value)


## Subject-to-subject comparison (4 topomap in orizzontale)



In [6]:
import json
import numpy as np
import torch
import mne
import matplotlib.pyplot as plt

from pathlib import Path
from ipywidgets import Dropdown, IntSlider, HBox, VBox, Output
from IPython.display import display, clear_output

project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

# ---- scegli soggetti ----
S1 = 0
S2 = 1

p1 = project_root / "data/processed/subject_tensors/subject_tensors_time" / f"subject_{S1:02d}.pt"
p2 = project_root / "data/processed/subject_tensors/subject_tensors_time" / f"subject_{S2:02d}.pt"

o1 = torch.load(p1, map_location="cpu")
o2 = torch.load(p2, map_location="cpu")

X1 = o1["X"].numpy()
y1 = o1["y"].numpy()
X2 = o2["X"].numpy()
y2 = o2["y"].numpy()

feature_cols = o1["feature_cols"]
ch_names1 = o1["ch_names"]
ch_names2 = o2["ch_names"]

assert feature_cols == o2["feature_cols"], "feature_cols diversi tra soggetti"
assert ch_names1 == ch_names2, "ch_names diversi / ordine diverso tra soggetti"
ch_names = ch_names1

n_windows = X1.shape[1]

# ---- label mapping (word) ----
label2idx_path = project_root / "data/interim/label2idx.json"
with open(label2idx_path, "r") as f:
    label2idx = json.load(f)
idx2label = {int(v): k for k, v in label2idx.items()}

# parole disponibili in ENTRAMBI i soggetti
common_classes = sorted(list(set(np.unique(y1)).intersection(set(np.unique(y2)))))
words = [idx2label.get(c, f"UNK_{c}") for c in common_classes]

# ---- MNE info ----
eloc_path = project_root / "src/io/ebneuro.locs"
montage = mne.channels.read_custom_montage(str(eloc_path))

info = mne.create_info(ch_names=list(ch_names), sfreq=o1.get("fs", 256), ch_types="eeg")
pos = montage.get_positions()
ch_pos = {k: v for k, v in pos["ch_pos"].items() if k in set(ch_names)}
montage_keep = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")
info.set_montage(montage_keep)

# ---- widgets ----
feat_dd = Dropdown(options=feature_cols, value=feature_cols[0], description="feature")
word_dd = Dropdown(options=list(zip(words, common_classes)), value=common_classes[0], description="word")
win_sl  = IntSlider(value=0, min=0, max=n_windows-1, step=1, description="window")
out = Output()

def mean_topo(X, y, cls_id, window, fidx):
    idx = np.where(y == cls_id)[0]
    if len(idx) == 0:
        return None
    return X[idx, window, :, fidx].mean(axis=0)

def draw(window, feature, cls_id):
    with out:
        clear_output(wait=True)

        fidx = feature_cols.index(feature)

        v1 = mean_topo(X1, y1, cls_id, window, fidx)
        v2 = mean_topo(X2, y2, cls_id, window, fidx)
        if v1 is None or v2 is None:
            print("missing trials for this word in one subject")
            return

        diff = v2 - v1
        absdiff = np.abs(diff)

        # similarità in [0,1]
        max_absdiff = float(absdiff.max())
        sim = 1.0 - (absdiff / (max_absdiff + 1e-12))

        # scale coerenti
        vmax_main = float(np.max(np.abs(np.concatenate([v1, v2]))))
        vmax_diff = float(np.max(np.abs(diff)))

        fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=150)

        im1, _ = mne.viz.plot_topomap(
            v1, info, axes=axes[0], show=False,
            sensors=True, contours=0, outlines="head",
            image_interp="cubic", vlim=(-vmax_main, vmax_main)
        )
        axes[0].set_title(f"Subject {S1:02d}", fontsize=13)

        im2, _ = mne.viz.plot_topomap(
            v2, info, axes=axes[1], show=False,
            sensors=True, contours=0, outlines="head",
            image_interp="cubic", vlim=(-vmax_main, vmax_main)
        )
        axes[1].set_title(f"Subject {S2:02d}", fontsize=13)

        im3, _ = mne.viz.plot_topomap(
            diff, info, axes=axes[2], show=False,
            sensors=True, contours=0, outlines="head",
            image_interp="cubic", vlim=(-vmax_diff, vmax_diff)
        )
        axes[2].set_title(f"Diff (S{S2:02d} − S{S1:02d})", fontsize=13)

        im4, _ = mne.viz.plot_topomap(
            sim, info, axes=axes[3], show=False,
            sensors=True, contours=0, outlines="head",
            image_interp="cubic", vlim=(0.0, 1.0)
        )
        axes[3].set_title("Similarity (1 − |diff|/max)", fontsize=13)

        # colorbar: una per S1/S2, una per diff, una per similarity
        cbar1 = plt.colorbar(im2, ax=axes[:2], fraction=0.046, pad=0.04)
        cbar1.set_label(feature)

        cbar2 = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
        cbar2.set_label("difference")

        cbar3 = plt.colorbar(im4, ax=axes[3], fraction=0.046, pad=0.04)
        cbar3.set_label("similarity (0-1)")

        word = idx2label.get(cls_id, f"UNK_{cls_id}")
        fig.suptitle(f"window={window} | word={word} | feature={feature}", fontsize=14, y=1.03)
        plt.show()

def _on_change(change=None):
    draw(win_sl.value, feat_dd.value, word_dd.value)

win_sl.observe(_on_change, names="value")
feat_dd.observe(_on_change, names="value")
word_dd.observe(_on_change, names="value")

ui = HBox([win_sl, feat_dd, word_dd])
display(VBox([ui, out]))

draw(win_sl.value, feat_dd.value, word_dd.value)

# ============================================================
# TOPOMAP VISUALIZATION — AGGREGATED FEATURES (NO TIME)
# ============================================================

In [7]:
# ---- carica tensore aggregato ----

SUBJECT_ID = 0

agg_path = project_root / "data" / "processed" / "subject_tensors" / "subject_tensors_aggregated_epoch" / f"subject_{SUBJECT_ID:02d}.pt"

data = torch.load(agg_path, map_location="cpu")

X_agg = data["X"].numpy()     # (epochs, 59, 40)
y = data["y"].numpy()

feature_cols = data["feature_cols"]
ch_names = data["ch_names"]

print("X_agg:", X_agg.shape)
print("num classes:", len(np.unique(y)))
print("num channels:", len(ch_names))

import mne

sfreq = 256

info = mne.create_info(
    ch_names=ch_names,
    sfreq=sfreq,
    ch_types="eeg"
)

montage = mne.channels.read_custom_montage(
    project_root / "src/io/ebneuro.locs"
)

info.set_montage(montage)

X_agg: (550, 59, 40)
num classes: 110
num channels: 59


<Info | 8 non-empty values
 bads: []
 ch_names: AF7, AF3, Fp1, FP2, AF4, AF8, F7, F5, F3, F1, F2, F4, F6, F8, ...
 chs: 59 EEG
 custom_ref_applied: False
 dig: 62 items (3 Cardinal, 59 EEG)
 highpass: 0.0 Hz
 lowpass: 128.0 Hz
 meas_date: unspecified
 nchan: 59
 projs: []
 sfreq: 256.0 Hz
>

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, VBox, interactive_output
from IPython.display import display

classes = sorted(np.unique(y).tolist())

feat_dd = Dropdown(options=feature_cols, value=feature_cols[0], description="feature")
class_dd = Dropdown(options=classes, value=classes[0], description="class")

def draw_agg(feature, cls):
    fidx = feature_cols.index(feature)
    idx = np.where(y == cls)[0]
    if len(idx) == 0:
        print("no trials for class", cls)
        return

    vals = X_agg[idx, :, fidx].mean(axis=0)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
    im, _ = mne.viz.plot_topomap(
        vals,
        info,
        axes=ax,
        show=False,
        sensors=True,
        contours=0,
        outlines="head",
        image_interp="cubic",
    )
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(feature)

    ax.set_title(f"Subject {SUBJECT_ID:02d} | class={cls} | {feature} | aggregated", fontsize=13)
    plt.show()

io = interactive_output(draw_agg, {"feature": feat_dd, "cls": class_dd})

display(VBox([VBox([feat_dd, class_dd]), io]))